# Kandinsky Game→Audio — Architecture Step 1: Game Event Encoder

This notebook builds the first part of an architecture that connects structured events from a game engine to Kandinsky's audio stream.

**In this step:** inspect the actual CrossDiT insertion points in the official Kandinsky source and implement a `GameEventEncoder` that creates separate game-event tokens without turning physics values into prompt text.

**Not included yet:** downloading Kandinsky weights, connecting cross-attention to the model, training, or generating audio. The tokens produced here come from randomly initialized parameters; they are not evidence of learned physical control.

The initial scope is **impact events only**. We will establish the data path first, then add real audio→game cross-attention.

## What we will do in this step

1. Verify where a game condition could be added to Kandinsky's audio stream.
2. Split a game-engine event into fields: event type, materials, measured physical values, and timing.
3. Create embeddings for categorical fields and masked, normalized value tokens for numeric fields.
4. Combine one summary token and the field tokens into a sequence for each event. In a later step, audio tokens will read this sequence through cross-attention.

Normalization statistics for numeric values must be calculated from the training data. This notebook does not invent means or standard deviations.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch
from torch import nn
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("No checkpoint is downloaded and no GPU training is performed in this step.")

## 1. Inspect the official Kandinsky source at a pinned revision

We fetch source code only; no model weights are downloaded. Pinning the commit keeps the inspected blocks stable even if the upstream repository changes.

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-arch-lab")
if not K6_DIR.exists():
    subprocess.run(["git","clone","--depth","1","--branch","main",
                    "https://github.com/kandinskylab/kandinsky-6.git",str(K6_DIR)],check=True)
current = subprocess.check_output(["git","-C",str(K6_DIR),"rev-parse","HEAD"],text=True).strip()
if current != K6_COMMIT:
    subprocess.run(["git","-C",str(K6_DIR),"fetch","--depth","1","origin",K6_COMMIT],check=True)
    subprocess.run(["git","-C",str(K6_DIR),"checkout","--detach",K6_COMMIT],check=True)
print("Kandinsky source commit:",subprocess.check_output(["git","-C",str(K6_DIR),"rev-parse","HEAD"],text=True).strip())

In [ ]:
dit_path = K6_DIR / "kandinsky/core/components/dit.py"
lines = dit_path.read_text(encoding="utf-8").splitlines()
anchors = [
    "class MultiheadCrossAttention",
    "class FusedTransformerDecoderBlock",
    "aud_out_t",
    "# V→A and A→V attention",
    "# ---- FFN ----",
]
for anchor in anchors:
    hits = [i for i,line in enumerate(lines) if anchor in line]
    print("\n###", anchor)
    for i in hits[:2]:
        lo, hi = max(0,i-2), min(len(lines),i+5)
        print("\n".join(f"{j+1}: {lines[j]}" for j in range(lo,hi)))

### What we found

- `MultiheadCrossAttention` can accept query tokens and conditioning tokens with different feature dimensions.
- `FusedTransformerDecoderBlock` runs self-attention and text cross-attention in the audio stream; when video is present, it also applies video↔audio attention.
- A natural place for the new condition is in the audio stream, after text attention: **audio queries ← game-event keys/values**. Game conditioning should also work when no video is supplied.

This insertion point requires changes to Kandinsky's existing code. The current generation notebook does not accept this input.

## 2. Event-token contract

The first example is an impact. For each event, we will create:

- One summary token: event type and the event's time interval.
- Two material tokens: contacting surface A and surface B.
- Eight numeric tokens: impulse, relative normal speed, the two object masses, contact duration, and the X/Y/Z dimensions of the bounding box.

Missing measurements are not represented as zero. We also create a mask for each numeric field; when the mask is off, later attention will ignore that field. We do not treat `simulation_assigned` values in the example config as real measurements; the demo below deliberately leaves the physics values empty.

In [ ]:
MATERIAL_TO_ID = {"wood":0,"metal":1,"glass":2,"ceramic":3,"concrete":4,"unknown":5}
EVENT_TO_ID = {"impact":0}  # Impact events only in this first step
NUMERIC_FIELDS = [
    "impulse_ns", "normal_relative_speed_m_s", "mass_a_kg", "mass_b_kg",
    "contact_duration_s", "bbox_x_m", "bbox_y_m", "bbox_z_m",
]
TOKEN_LAYOUT = ["event_summary","material_a","material_b",*NUMERIC_FIELDS]
print("Tokens per event:",len(TOKEN_LAYOUT))
for i,name in enumerate(TOKEN_LAYOUT):
    print(f"{i:02d}  {name}")

In [ ]:
class GameEventEncoder(nn.Module):
    """Encodes engine events as typed tokens without converting them to prompt text."""
    def __init__(self, d_model=256, n_event_types=1, n_materials=6, n_numeric_fields=8):
        super().__init__()
        self.d_model = d_model
        self.event_type = nn.Embedding(n_event_types, d_model)
        self.material_a = nn.Embedding(n_materials, d_model)
        self.material_b = nn.Embedding(n_materials, d_model)
        self.time_projection = nn.Sequential(nn.Linear(2, d_model), nn.SiLU(), nn.Linear(d_model,d_model))
        self.field_type = nn.Embedding(n_numeric_fields, d_model)
        # Input: scalar normalized using the training set, plus a measurement mask
        self.value_projection = nn.Sequential(nn.Linear(2,d_model),nn.SiLU(),nn.Linear(d_model,d_model))
        self.norm = nn.LayerNorm(d_model)

    def forward(self, event_type_ids, material_a_ids, material_b_ids,
                time_features, numeric_values_z, numeric_mask):
        # All tensors have the shape [batch, events, ...].
        time_emb = self.time_projection(time_features)
        summary = self.event_type(event_type_ids) + time_emb
        mat_a = self.material_a(material_a_ids) + time_emb
        mat_b = self.material_b(material_b_ids) + time_emb

        masked_values = torch.where(numeric_mask, numeric_values_z, torch.zeros_like(numeric_values_z))
        value_and_mask = torch.stack([masked_values, numeric_mask.float()], dim=-1)
        numeric_tokens = self.value_projection(value_and_mask)
        field_ids = torch.arange(numeric_values_z.shape[-1],device=numeric_values_z.device)
        numeric_tokens = numeric_tokens + self.field_type(field_ids)[None,None,:,:] + time_emb[:,:,None,:]

        # Order: summary, material A, material B, then one token per physical field.
        tokens = torch.cat([summary[:,:,None,:],mat_a[:,:,None,:],mat_b[:,:,None,:],numeric_tokens],dim=2)
        summary_valid = torch.ones_like(event_type_ids,dtype=torch.bool)[:,:,None]
        material_valid = torch.ones_like(material_a_ids,dtype=torch.bool)[:,:,None].expand(-1,-1,2)
        token_mask = torch.cat([summary_valid,material_valid,numeric_mask],dim=2)
        batch, n_events, n_tokens, dim = tokens.shape
        return tokens.reshape(batch,n_events*n_tokens,dim), token_mask.reshape(batch,n_events*n_tokens)

## 3. Shape demonstration

The material and event type are present below, while physical measurements are **deliberately missing**. This cell demonstrates the encoder's token format; it has not been trained. Therefore, the embeddings should not be expected to have any meaning for audio.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = GameEventEncoder().to(device)

# [batch=1, event=1] — wood/concrete impact
event_type_ids = torch.tensor([[EVENT_TO_ID["impact"]]],device=device)
material_a_ids = torch.tensor([[MATERIAL_TO_ID["wood"]]],device=device)
material_b_ids = torch.tensor([[MATERIAL_TO_ID["concrete"]]],device=device)

# Timing features: a demo event starting at the beginning of a 1-second target window.
# In a real task, divide by WINDOW_SECONDS using video/game timing.
time_features = torch.tensor([[[0.0,0.2]]],device=device)

# We do not invent physics values: normalized values are masked and invalid.
numeric_values_z = torch.zeros((1,1,len(NUMERIC_FIELDS)),device=device)
numeric_mask = torch.zeros_like(numeric_values_z,dtype=torch.bool)

with torch.no_grad():
    game_tokens, game_token_mask = encoder(
        event_type_ids,material_a_ids,material_b_ids,
        time_features,numeric_values_z,numeric_mask
    )
print("Game-token shape:",tuple(game_tokens.shape))
print("Token-mask shape:",tuple(game_token_mask.shape))
print("Valid token count:",int(game_token_mask.sum().item()),"/",game_token_mask.numel())
print("Token order:",TOKEN_LAYOUT)

## Step output and next step

The encoder is not connected to Kandinsky yet. For now, it converts a game event into a masked, typed token sequence that is separate from the prompt.

**Next step:** write a `Game→Audio` attention adapter that connects the encoder output to the audio stream inside Kandinsky's real `FusedTransformerDecoderBlock`. The new adapter's gate will start at zero: when closed, the base model should retain its behavior; when opened, it should be able to use information from the game tokens. We will not begin training until we have paired audio and real physics values.

For now, run only this notebook in Colab: source inspection and a token-shape demonstration. **No checkpoint download, audio generation, or model training.**